# Tests — utils_billing

> Unit and integration tests for the billing orchestrator.

In [ ]:
from fh_saas.utils_billing import (
    SubscriptionStatus, BillingConfig, validate_billing_config,
    AccessDecision, resolve_subscription_access, _should_apply_status_change,
    init_trial_on_first_login, resolve_checkout_return,
)
from dataclasses import dataclass
from datetime import datetime, timedelta

## SubscriptionStatus.normalize

In [ ]:
assert SubscriptionStatus.normalize('active') == SubscriptionStatus.active
assert SubscriptionStatus.normalize('ACTIVE') == SubscriptionStatus.active
assert SubscriptionStatus.normalize('  Active ') == SubscriptionStatus.active
print('\u2705 normalize: standard values')

In [ ]:
assert SubscriptionStatus.normalize('cancelled') == SubscriptionStatus.canceled
assert SubscriptionStatus.normalize('incomplete') == SubscriptionStatus.past_due
assert SubscriptionStatus.normalize('unpaid') == SubscriptionStatus.past_due
print('\u2705 normalize: aliases')

In [ ]:
assert SubscriptionStatus.normalize('') == SubscriptionStatus.none
assert SubscriptionStatus.normalize(None) == SubscriptionStatus.none
assert SubscriptionStatus.normalize('garbage_xyz') == SubscriptionStatus.none
print('\u2705 normalize: edge cases')

## validate_billing_config

In [ ]:
valid = BillingConfig(
    stripe_secret_key='sk_test_123',
    stripe_webhook_secret='whsec_test',
    monthly_price_id='price_monthly',
)
assert validate_billing_config(valid) == []
print('\u2705 validate: valid config')

In [ ]:
empty = BillingConfig()
errs = validate_billing_config(empty)
assert any('STRIPE_SECRET_KEY' in e for e in errs)
assert any('STRIPE_WEBHOOK_SECRET' in e for e in errs)
assert any('price ID' in e for e in errs)
print(f'\u2705 validate: empty config -> {len(errs)} errors')

In [ ]:
dev = BillingConfig(
    stripe_secret_key='sk_test_123',
    monthly_price_id='price_monthly',
    is_development=True,
)
errs = validate_billing_config(dev)
assert errs == [], f'Dev mode should skip webhook secret check, got: {errs}'
print('\u2705 validate: dev mode skips webhook secret')

## resolve_subscription_access

In [ ]:
cfg = BillingConfig(
    stripe_secret_key='sk_test',
    monthly_price_id='price_m',
    pricing_path='/pricing',
    portal_path='/billing-portal',
    success_path='/payment-success',
)

@dataclass
class MockSub:
    status: str = 'active'
    trial_end: str = None
    current_period_end: str = None

# active -> allowed
d = resolve_subscription_access(MockSub(status='active'), cfg)
assert d.allowed and d.reason == SubscriptionStatus.active
print('\u2705 access: active')

In [ ]:
# none sub -> denied
d = resolve_subscription_access(None, cfg)
assert not d.allowed and d.redirect_to == '/pricing'
print('\u2705 access: no subscription')

In [ ]:
# trialing, not expired
future = (datetime.utcnow() + timedelta(days=15)).isoformat()
d = resolve_subscription_access(MockSub(status='trialing', trial_end=future), cfg)
assert d.allowed and d.reason == SubscriptionStatus.trialing
print('\u2705 access: trialing (active trial)')

In [ ]:
# trialing, expired
past = (datetime.utcnow() - timedelta(days=1)).isoformat()
d = resolve_subscription_access(MockSub(status='trialing', trial_end=past), cfg)
assert not d.allowed and d.redirect_to == '/pricing'
print('\u2705 access: trialing (expired)')

In [ ]:
# past_due within grace
grace_end = (datetime.utcnow() + timedelta(days=1)).isoformat()
d = resolve_subscription_access(MockSub(status='past_due', current_period_end=grace_end), cfg)
assert d.allowed
print('\u2705 access: past_due within grace')

In [ ]:
# past_due beyond grace
old = (datetime.utcnow() - timedelta(days=30)).isoformat()
d = resolve_subscription_access(MockSub(status='past_due', current_period_end=old), cfg)
assert not d.allowed and d.redirect_to == '/billing-portal'
print('\u2705 access: past_due beyond grace')

In [ ]:
# canceled -> denied
d = resolve_subscription_access(MockSub(status='canceled'), cfg)
assert not d.allowed and d.redirect_to == '/pricing'
print('\u2705 access: canceled')

## _should_apply_status_change

In [ ]:
# canceled always applies
assert _should_apply_status_change('active', 'canceled', None, None) is True
print('\u2705 guard: canceled always applies')

In [ ]:
# active -> trialing blocked
assert _should_apply_status_change('active', 'trialing', None, None) is False
print('\u2705 guard: active -> trialing blocked')

In [ ]:
# stale period_end blocked
old_period = '2024-06-01T00:00:00'
new_period = '2024-05-01T00:00:00'
assert _should_apply_status_change('active', 'active', old_period, new_period) is False
print('\u2705 guard: stale period_end blocked')

In [ ]:
# newer period_end applies
old_period = '2024-06-01T00:00:00'
new_period = '2024-07-01T00:00:00'
assert _should_apply_status_change('active', 'active', old_period, new_period) is True
print('\u2705 guard: newer period_end applies')

## init_trial_on_first_login

In [ ]:
class MockSubscriptionStore:
    def __init__(self):
        self._store = {}
        self._events = set()
    def get_subscription(self, tenant_id):
        return self._store.get(tenant_id)
    def upsert_subscription(self, **fields):
        tid = fields.get('tenant_id', '')
        self._store[tid] = type('Sub', (), fields)()
    def has_processed_event(self, event_id):
        return event_id in self._events
    def record_event(self, event_id, event_type, status, payload_json=''):
        self._events.add(event_id)

store = MockSubscriptionStore()
trial_cfg = BillingConfig(stripe_secret_key='sk_test', monthly_price_id='pm', trial_days=30)
result = init_trial_on_first_login('tenant_1', 'user@test.com', store, trial_cfg)
assert result is not None
assert result.status == 'trialing'
print('\u2705 init_trial: creates trial')

In [ ]:
# idempotent: second call returns None
result2 = init_trial_on_first_login('tenant_1', 'user@test.com', store, trial_cfg)
assert result2 is None
print('\u2705 init_trial: idempotent')

## resolve_checkout_return

In [ ]:
# active sub -> returns active
store2 = MockSubscriptionStore()
store2.upsert_subscription(tenant_id='t1', stripe_sub_id='sub_1', status='active')
s = resolve_checkout_return('sess_1', 't1', store2)
assert s == SubscriptionStatus.active
print('\u2705 checkout_return: active')

In [ ]:
# no sub -> checkout_pending
store3 = MockSubscriptionStore()
s = resolve_checkout_return('sess_2', 't_new', store3)
assert s == SubscriptionStatus.checkout_pending
print('\u2705 checkout_return: pending')

## Webhook Idempotency

In [ ]:
store4 = MockSubscriptionStore()
store4.record_event('evt_1', 'test', 'processed')
assert store4.has_processed_event('evt_1') is True
assert store4.has_processed_event('evt_2') is False
print('\u2705 idempotency: record & check')

---
All tests passed! ✅